In [1]:
"""ai-tests notebook

Goal:
- Validate matcher scoring inside a Jupyter notebook, using ONLY:
    sentence-transformers/all-MiniLM-L6-v2

Assumptions:
- You are running this notebook using the conda env kernel: `conda activate ai`
- You install Python deps into that env (Cell 1)

Run order:
- Cell 0: repo + import-path setup (this cell)
- Cell 1: install notebook requirements (into the active conda env)
- Cell 2+: run eval / sanity checks

Notes:
- The first model call may download Hugging Face weights.
- CUDA will be used automatically if available.
"""

from __future__ import annotations

import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Find repo root by locating the `ai-tests/` directory."""
    start = start.resolve()
    for p in (start, *start.parents):
        if (p / "ai-tests").is_dir():
            return p
    raise RuntimeError(f"Could not find repo root above {start} (missing ai-tests/)")


REPO_ROOT = find_repo_root(Path.cwd())
AI_TESTS = REPO_ROOT / "ai-tests"
DATASET = AI_TESTS / "evals" / "dataset_smoke.json"

# Allow imports from `ai-tests/`.
if str(AI_TESTS) not in sys.path:
    sys.path.insert(0, str(AI_TESTS))

print("repo:", REPO_ROOT)
print("ai-tests:", AI_TESTS)
print("dataset:", DATASET, "exists=", DATASET.exists())
print("python:", sys.executable)
print("CONDA_PREFIX:", os.environ.get("CONDA_PREFIX"))
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("model:", "sentence-transformers/all-MiniLM-L6-v2")
print("next: run Cell 1 to install requirements into this env")


repo: /home/uranium/personal/SeekersConnect
ai-tests: /home/uranium/personal/SeekersConnect/ai-tests
dataset: /home/uranium/personal/SeekersConnect/ai-tests/evals/dataset_smoke.json exists= True
python: /home/uranium/.conda/envs/ai/bin/python3.10
CONDA_PREFIX: /home/uranium/.conda/envs/ai
CUDA_VISIBLE_DEVICES: None
model: sentence-transformers/all-MiniLM-L6-v2
next: run Cell 1 to install requirements into this env


In [2]:
# Install notebook dependencies into the active conda env (run once per env)
#
# You said you're using:
#   conda activate ai
# Make sure this notebook is using that same kernel/environment.

import sys

req = REPO_ROOT / "ai-tests" / "requirements-matcher.txt"

!{sys.executable} -m pip install -U pip
!{sys.executable} -m pip install -r "{req}"
!{sys.executable} -m pip install -U sentence-transformers

print("installed requirements from:", req)
print("python:", sys.executable)


installed requirements from: /home/uranium/personal/SeekersConnect/ai-tests/requirements-matcher.txt
python: /home/uranium/.conda/envs/ai/bin/python3.10


In [4]:
# Run the offline eval (recall@K, MRR) on the smoke dataset
# using sentence-transformers/all-MiniLM-L6-v2.

from __future__ import annotations

import json
import math

import torch
from sentence_transformers import SentenceTransformer


def _sigmoid(x: float) -> float:
    if x >= 0:
        z = math.exp(-x)
        return 1.0 / (1.0 + z)
    z = math.exp(x)
    return z / (1.0 + z)


def recall_at_k(ranked_ids: list[str], positive_ids: set[str], k: int) -> float:
    if not positive_ids:
        return 1.0
    top = set(ranked_ids[:k])
    return len(top & positive_ids) / len(positive_ids)


def mrr(ranked_ids: list[str], positive_ids: set[str]) -> float:
    if not positive_ids:
        return 1.0
    for i, jid in enumerate(ranked_ids, start=1):
        if jid in positive_ids:
            return 1.0 / i
    return 0.0


MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_ID)

# Prefer GPU if available (SentenceTransformer handles device internally too)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)


data = json.loads(DATASET.read_text(encoding="utf-8"))
cases = data.get("cases") or []

results = []
for case in cases:
    resume = case["resume_text"].strip()
    k = int(case.get("k", 25))
    candidates = case["candidates"]

    texts = [resume] + [c["text"].strip() for c in candidates]

    embs = model.encode(
        texts,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    r = embs[0:1]
    job_embs = embs[1:]

    sims = (r @ job_embs.T).squeeze(0)
    scores = [_sigmoid(float(sims[i].item())) for i in range(len(candidates))]

    order = sorted(range(len(candidates)), key=lambda i: scores[i], reverse=True)
    ranked_ids = [candidates[i]["id"] for i in order]

    positive_ids = {c["id"] for c in candidates if c.get("relevant")}

    results.append(
        {
            "case_id": case.get("id"),
            "recall_at_k": recall_at_k(ranked_ids, positive_ids, k),
            "mrr": mrr(ranked_ids, positive_ids),
            "k": k,
            "ranked_ids": ranked_ids,
            "top_score": scores[order[0]] if order else None,
        }
    )

mean_recall = sum(r["recall_at_k"] for r in results) / len(results)
mean_mrr = sum(r["mrr"] for r in results) / len(results)

print(
    json.dumps(
        {
            "model": MODEL_ID,
            "device": device,
            "per_case": results,
            "mean_recall_at_k": mean_recall,
            "mean_mrr": mean_mrr,
        },
        indent=2,
    )
)


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 10015.38it/s]


{
  "model": "sentence-transformers/all-MiniLM-L6-v2",
  "device": "cpu",
  "per_case": [
    {
      "case_id": "python_vs_unrelated",
      "recall_at_k": 1.0,
      "mrr": 1.0,
      "k": 5,
      "ranked_ids": [
        "j1",
        "j4",
        "j3",
        "j2"
      ],
      "top_score": 0.6748429193866597
    }
  ],
  "mean_recall_at_k": 1.0,
  "mean_mrr": 1.0
}


In [ ]:
# Run the eval against a more complex dataset

from __future__ import annotations

import json
import math
from pathlib import Path

import torch
from sentence_transformers import SentenceTransformer


def _sigmoid(x: float) -> float:
    if x >= 0:
        z = math.exp(-x)
        return 1.0 / (1.0 + z)
    z = math.exp(x)
    return z / (1.0 + z)


def recall_at_k(ranked_ids: list[str], positive_ids: set[str], k: int) -> float:
    if not positive_ids:
        return 1.0
    top = set(ranked_ids[:k])
    return len(top & positive_ids) / len(positive_ids)


def mrr(ranked_ids: list[str], positive_ids: set[str]) -> float:
    if not positive_ids:
        return 1.0
    for i, jid in enumerate(ranked_ids, start=1):
        if jid in positive_ids:
            return 1.0 / i
    return 0.0


def run_dataset(dataset_path: Path, model_id: str = "sentence-transformers/all-MiniLM-L6-v2") -> dict:
    model = SentenceTransformer(model_id)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    data = json.loads(dataset_path.read_text(encoding="utf-8"))
    cases = data.get("cases") or []

    results = []
    for case in cases:
        resume = case["resume_text"].strip()
        k = int(case.get("k", 25))
        candidates = case["candidates"]

        texts = [resume] + [c["text"].strip() for c in candidates]
        embs = model.encode(
            texts,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        r = embs[0:1]
        job_embs = embs[1:]
        sims = (r @ job_embs.T).squeeze(0)
        scores = [_sigmoid(float(sims[i].item())) for i in range(len(candidates))]

        order = sorted(range(len(candidates)), key=lambda i: scores[i], reverse=True)
        ranked_ids = [candidates[i]["id"] for i in order]
        positive_ids = {c["id"] for c in candidates if c.get("relevant")}

        results.append(
            {
                "case_id": case.get("id"),
                "recall_at_k": recall_at_k(ranked_ids, positive_ids, k),
                "mrr": mrr(ranked_ids, positive_ids),
                "k": k,
                "ranked_ids": ranked_ids,
                "top_score": scores[order[0]] if order else None,
                "positives": sorted(list(positive_ids)),
            }
        )

    mean_recall = sum(r["recall_at_k"] for r in results) / len(results)
    mean_mrr = sum(r["mrr"] for r in results) / len(results)

    return {
        "model": model_id,
        "device": device,
        "dataset": str(dataset_path),
        "cases": len(results),
        "mean_recall_at_k": mean_recall,
        "mean_mrr": mean_mrr,
        "per_case": results,
    }


DATASET_COMPLEX = AI_TESTS / "evals" / "dataset_complex.json"
print("dataset:", DATASET_COMPLEX, "exists=", DATASET_COMPLEX.exists())
print(json.dumps(run_dataset(DATASET_COMPLEX), indent=2))


In [5]:
# Generate a deep + long dataset (120 cases) and evaluate MiniLM on it

from __future__ import annotations

import json
import math
import random
from pathlib import Path

import torch
from sentence_transformers import SentenceTransformer

MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
OUT_PATH = AI_TESTS / "evals" / "dataset_deep_120.json"


def _sigmoid(x: float) -> float:
    if x >= 0:
        z = math.exp(-x)
        return 1.0 / (1.0 + z)
    z = math.exp(x)
    return z / (1.0 + z)


def recall_at_k(ranked_ids: list[str], positive_ids: set[str], k: int) -> float:
    if not positive_ids:
        return 1.0
    top = set(ranked_ids[:k])
    return len(top & positive_ids) / len(positive_ids)


def mrr(ranked_ids: list[str], positive_ids: set[str]) -> float:
    if not positive_ids:
        return 1.0
    for i, jid in enumerate(ranked_ids, start=1):
        if jid in positive_ids:
            return 1.0 / i
    return 0.0


def _long_text(paragraphs: list[str], repeat: int = 3) -> str:
    # Make text deep/long while staying readable.
    return "\n\n".join(paragraphs * repeat)


def build_deep_dataset(n_cases: int = 120, seed: int = 1337) -> dict:
    rng = random.Random(seed)

    # Profiles: (name, resume paragraphs, relevant job templates, hard-negative templates)
    profiles = [
        (
            "backend_python",
            [
                "Backend engineer with 6+ years in Python (FastAPI, Django) and Postgres. Built REST APIs, background workers, and internal tooling.",
                "Operational experience: Docker, CI/CD, incident response, observability (metrics/logs/traces).",
                "Worked with Redis caches, queues (Celery/RQ), and AWS (EC2, RDS, S3).",
                "Focus on performance, correctness, schema migrations, and pragmatic testing strategies.",
            ],
            [
                "Backend engineer — Python, FastAPI, Postgres, Redis. Own APIs end-to-end.",
                "Senior backend engineer — Python services, SQL, distributed systems, message queues.",
            ],
            [
                "Frontend engineer — React, TypeScript, CSS, design systems.",
                "Line cook — fast-paced kitchen, nights/weekends.",
                "Accountant — month-end close, reconciliation, audits.",
            ],
        ),
        (
            "frontend_react",
            [
                "Frontend engineer specializing in React and TypeScript. Built component libraries and accessible UI flows.",
                "Performance work: code-splitting, memoization, bundle analysis, image optimization.",
                "Testing: Playwright, Jest, CI pipelines; strong collaboration with designers in Figma.",
                "Experience with Next.js, routing, server rendering, and API integration.",
            ],
            [
                "Frontend engineer — React, TypeScript, Next.js, accessibility.",
                "UI engineer — design systems, Storybook, React, CSS-in-JS.",
            ],
            [
                "Data engineer — Airflow, dbt, Spark, warehouses.",
                "DevOps engineer — Kubernetes, Terraform, on-call.",
                "Warehouse associate — packing and shipping.",
            ],
        ),
        (
            "data_engineer",
            [
                "Data engineer building ETL/ELT in Python with Airflow, dbt, and Spark. Strong SQL.",
                "Worked with S3, warehouses (Redshift/Snowflake-like), and data modeling for analytics.",
                "Batch pipelines, incremental loads, data quality checks, and backfills.",
                "Some streaming exposure (Kafka) and orchestration best practices.",
            ],
            [
                "Data engineer — Airflow, dbt, Python, Spark, data warehouses.",
                "Senior data engineer — Spark, streaming, Kafka, Airflow, SQL.",
            ],
            [
                "Backend engineer — Go microservices, gRPC, Kubernetes.",
                "Graphic designer — Figma, branding, marketing assets.",
                "Sales development rep — outbound and lead qualification.",
            ],
        ),
        (
            "platform_sre",
            [
                "Platform/SRE engineer operating Kubernetes clusters and building Terraform modules.",
                "Observability: Prometheus, Grafana, alerting, dashboards, SLOs, incident response.",
                "CI/CD, container security basics, and runtime debugging. On-call rotations.",
                "Cloud networking fundamentals and reliability improvements.",
            ],
            [
                "Site reliability engineer — Kubernetes, Terraform, observability, on-call.",
                "Platform engineer — Kubernetes, CI/CD, monitoring, cloud networking.",
            ],
            [
                "Frontend engineer — React, TypeScript.",
                "Customer support — tickets and troubleshooting.",
                "Cook — kitchen staff.",
            ],
        ),
        (
            "ml_embeddings",
            [
                "Software engineer with applied ML experience: embeddings, semantic search, retrieval evaluation, and offline/online metrics.",
                "Built text similarity pipelines and data tooling; comfortable with Python APIs and Postgres.",
                "Experience with model deployment constraints, batching, and latency profiling.",
                "Pragmatic: focuses on testability, iteration speed, and monitoring.",
            ],
            [
                "Machine learning engineer — embeddings, retrieval, evaluation, PyTorch.",
                "Backend engineer — Python APIs, Postgres, async workers (plus semantic search exposure).",
            ],
            [
                "Recruiter — sourcing and interviews.",
                "IT helpdesk — resets, ticket triage.",
                "Warehouse associate — packing and shipping.",
            ],
        ),
    ]

    # Deep job description template with extra noise + requirements sections.
    def job_text(title: str, domain_noise: str, must_have: list[str], nice_to_have: list[str]) -> str:
        base = [
            f"Role: {title}",
            "\nAbout the team:\nWe build product features with a focus on reliability, collaboration, and measurable outcomes.",
            f"\nContext:\n{domain_noise}",
            "\nResponsibilities:\n- Own projects end-to-end\n- Write tests and ship incrementally\n- Work with stakeholders\n- Improve performance and reliability",
            "\nRequirements (must have):\n- " + "\n- ".join(must_have),
            "\nNice to have:\n- " + "\n- ".join(nice_to_have),
            "\nInterview process:\nPhone screen, technical, system design/collaboration round, and final.",
        ]
        # Make it long.
        return _long_text(base, repeat=2)

    def resume_text(profile_paras: list[str], extra: str) -> str:
        base = profile_paras + [
            "\nProjects:\n- Migrated legacy service safely with feature flags\n- Improved latency by profiling and caching\n- Wrote docs/runbooks and automated repetitive tasks",
            f"\nAdditional context:\n{extra}",
        ]
        return _long_text(base, repeat=3)

    # Domain noise paragraphs (adds length + realism without changing signal too much)
    noise_bank = [
        "We coordinate across time zones and value clear written communication. The product handles high read throughput and occasional burst traffic.",
        "We care about code review quality, observability, and operational readiness. Releases are frequent and incremental.",
        "We run experiments, measure success, and iterate. Security and privacy considerations apply to user data.",
        "We prefer boring technology where possible, but are open to improvements backed by evidence.",
    ]

    cases = []
    for i in range(n_cases):
        prof = profiles[i % len(profiles)]
        name, resume_paras, rel_titles, hard_negs = prof

        extra_noise = " ".join(rng.sample(noise_bank, k=2))
        resume = resume_text(resume_paras, extra_noise)

        # Build 12 candidates: 2 relevant + 10 negatives (some hard, some random from other profiles)
        candidates = []

        # relevant
        for rj, title in enumerate(rel_titles[:2], start=1):
            must = ["Strong communication", "Collaborative engineering", "Shipping mindset"]
            # Add profile-specific must-haves
            if name == "backend_python":
                must = ["Python", "REST APIs", "PostgreSQL", "Docker"] + must
                nice = ["Redis", "AWS", "Async workers"]
            elif name == "frontend_react":
                must = ["React", "TypeScript", "Accessibility"] + must
                nice = ["Next.js", "Storybook", "Playwright"]
            elif name == "data_engineer":
                must = ["SQL", "Python", "Airflow", "Data modeling"] + must
                nice = ["dbt", "Spark", "Kafka"]
            elif name == "platform_sre":
                must = ["Kubernetes", "Terraform", "Observability"] + must
                nice = ["SLOs", "Incident response", "Cloud networking"]
            else:
                must = ["Embeddings", "Retrieval", "Evaluation", "Python"] + must
                nice = ["PyTorch", "Vector search", "Batching/latency"]

            text = job_text(title, extra_noise, must, nice)
            candidates.append({"id": f"{name}_rel_{i}_{rj}", "text": text, "relevant": True})

        # negatives: 3 hard negatives from same profile's hard_negs
        for hn in hard_negs:
            must = ["Reliability", "Ownership", "Teamwork"]
            nice = ["Documentation", "Process improvements"]
            text = job_text(hn, "This role is unrelated to the resume's main domain.", must, nice)
            candidates.append({"id": f"{name}_hardneg_{i}_{abs(hash(hn))%9999}", "text": text, "relevant": False})

        # remaining negatives sampled from other profiles' relevant titles (semantic neighbors)
        other_profiles = [p for p in profiles if p[0] != name]
        rng.shuffle(other_profiles)
        while len(candidates) < 12:
            op = rng.choice(other_profiles)
            otitles = op[2]
            title = rng.choice(otitles)
            must = ["General software engineering", "Team collaboration"]
            nice = ["Nice-to-have skills"]
            text = job_text(title, "This looks plausible but does not match the resume well.", must, nice)
            candidates.append({"id": f"neg_{name}_{i}_{len(candidates)}", "text": text, "relevant": False})

        rng.shuffle(candidates)

        cases.append(
            {
                "id": f"deep_{i:03d}_{name}",
                "resume_text": resume,
                "k": 5,
                "candidates": candidates,
            }
        )

    return {"description": "Deep/long 120-case dataset for MiniLM matcher.", "cases": cases}


def eval_dataset(dataset: dict) -> dict:
    model = SentenceTransformer(MODEL_ID)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    results = []
    for case in dataset["cases"]:
        resume = case["resume_text"].strip()
        k = int(case.get("k", 25))
        candidates = case["candidates"]

        texts = [resume] + [c["text"].strip() for c in candidates]
        embs = model.encode(
            texts,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        r = embs[0:1]
        job_embs = embs[1:]
        sims = (r @ job_embs.T).squeeze(0)
        scores = [_sigmoid(float(sims[i].item())) for i in range(len(candidates))]

        order = sorted(range(len(candidates)), key=lambda i: scores[i], reverse=True)
        ranked_ids = [candidates[i]["id"] for i in order]
        positive_ids = {c["id"] for c in candidates if c.get("relevant")}

        results.append(
            {
                "case_id": case["id"],
                "recall_at_k": recall_at_k(ranked_ids, positive_ids, k),
                "mrr": mrr(ranked_ids, positive_ids),
                "positives": len(positive_ids),
                "candidates": len(candidates),
            }
        )

    mean_recall = sum(r["recall_at_k"] for r in results) / len(results)
    mean_mrr = sum(r["mrr"] for r in results) / len(results)

    return {
        "model": MODEL_ID,
        "device": device,
        "cases": len(results),
        "mean_recall_at_k": mean_recall,
        "mean_mrr": mean_mrr,
        "per_case": results,
    }


deep = build_deep_dataset(n_cases=120, seed=1337)
OUT_PATH.write_text(json.dumps(deep, ensure_ascii=False, indent=2), encoding="utf-8")

print("wrote:", OUT_PATH)
print("cases:", len(deep["cases"]))
print("first_resume_chars:", len(deep["cases"][0]["resume_text"]))
print("first_job_chars:", len(deep["cases"][0]["candidates"][0]["text"]))

summary = eval_dataset(deep)
print(json.dumps({k: summary[k] for k in ["model", "device", "cases", "mean_recall_at_k", "mean_mrr"]}, indent=2))

# Optional: inspect worst cases by MRR
worst = sorted(summary["per_case"], key=lambda r: r["mrr"])[:10]
print("worst_10:", json.dumps(worst, indent=2))


wrote: /home/uranium/personal/SeekersConnect/ai-tests/evals/dataset_deep_120.json
cases: 120
first_resume_chars: 2311
first_job_chars: 1656


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 7772.12it/s]


{
  "model": "sentence-transformers/all-MiniLM-L6-v2",
  "device": "cpu",
  "cases": 120,
  "mean_recall_at_k": 1.0,
  "mean_mrr": 1.0
}
worst_10: [
  {
    "case_id": "deep_000_backend_python",
    "recall_at_k": 1.0,
    "mrr": 1.0,
    "positives": 2,
    "candidates": 12
  },
  {
    "case_id": "deep_001_frontend_react",
    "recall_at_k": 1.0,
    "mrr": 1.0,
    "positives": 2,
    "candidates": 12
  },
  {
    "case_id": "deep_002_data_engineer",
    "recall_at_k": 1.0,
    "mrr": 1.0,
    "positives": 2,
    "candidates": 12
  },
  {
    "case_id": "deep_003_platform_sre",
    "recall_at_k": 1.0,
    "mrr": 1.0,
    "positives": 2,
    "candidates": 12
  },
  {
    "case_id": "deep_004_ml_embeddings",
    "recall_at_k": 1.0,
    "mrr": 1.0,
    "positives": 2,
    "candidates": 12
  },
  {
    "case_id": "deep_005_backend_python",
    "recall_at_k": 1.0,
    "mrr": 1.0,
    "positives": 2,
    "candidates": 12
  },
  {
    "case_id": "deep_006_frontend_react",
    "recall_at_k"

In [8]:
# Harder dataset (120 cases): more candidates + near-miss negatives + richer metrics

from __future__ import annotations

import json
import math
import random
from collections import Counter
from pathlib import Path

import torch
from sentence_transformers import SentenceTransformer

MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
OUT_PATH_HARD = AI_TESTS / "evals" / "dataset_hard_120.json"


def _sigmoid(x: float) -> float:
    if x >= 0:
        z = math.exp(-x)
        return 1.0 / (1.0 + z)
    z = math.exp(x)
    return z / (1.0 + z)


def recall_at_k(ranked_ids: list[str], positive_ids: set[str], k: int) -> float:
    if not positive_ids:
        return 1.0
    top = set(ranked_ids[:k])
    return len(top & positive_ids) / len(positive_ids)


def first_relevant_rank(ranked_ids: list[str], positive_ids: set[str]) -> int | None:
    for i, jid in enumerate(ranked_ids, start=1):
        if jid in positive_ids:
            return i
    return None


def mrr(ranked_ids: list[str], positive_ids: set[str]) -> float:
    r = first_relevant_rank(ranked_ids, positive_ids)
    return 1.0 / r if r else 0.0


def _long_text(parts: list[str], repeat: int = 2) -> str:
    return "\n\n".join(parts * repeat)


def build_hard_dataset(n_cases: int = 120, candidates_per_case: int = 50, seed: int = 2026) -> dict:
    rng = random.Random(seed)

    profiles = [
        (
            "backend_python",
            [
                "Backend engineer (Python) building APIs, background workers, and data-heavy endpoints.",
                "Skills: FastAPI/Django, Postgres, Redis, Docker, CI/CD, observability.",
                "Experience with caching, queues, schema design, migrations, and performance profiling.",
            ],
            [
                "Backend engineer — Python, FastAPI, Postgres, Redis.",
                "Senior backend engineer — Python services, SQL, message queues.",
            ],
            [
                # near-miss negatives: similar words but wrong focus
                "Backend engineer — Node.js, MongoDB, GraphQL, React.",
                "Data engineer — Python, SQL, Airflow, dbt (analytics pipelines).",
                "Platform engineer — Kubernetes, Terraform, monitoring, on-call.",
            ],
        ),
        (
            "frontend_react",
            [
                "Frontend engineer shipping React/TypeScript UIs with accessibility and performance focus.",
                "Built component libraries, design systems, and tested with Playwright/Jest.",
                "Next.js routing/rendering and integration with backend APIs.",
            ],
            [
                "Frontend engineer — React, TypeScript, Next.js, accessibility.",
                "UI engineer — design systems, Storybook, React.",
            ],
            [
                "Fullstack engineer — React + Node APIs + Postgres.",
                "Mobile engineer — React Native, Expo, releases.",
                "Backend engineer — Python APIs (no frontend focus).",
            ],
        ),
        (
            "data_engineer",
            [
                "Data engineer building ETL/ELT with Python and SQL.",
                "Orchestration with Airflow; modeling with dbt; compute with Spark.",
                "Warehouses, incremental loads, data quality checks, and backfills.",
            ],
            [
                "Data engineer — Airflow, dbt, Python, Spark, warehouses.",
                "Senior data engineer — Spark, Kafka, Airflow, SQL.",
            ],
            [
                "Data analyst — dashboards, Looker, SQL (no pipelines).",
                "Backend engineer — Go microservices, gRPC, Kubernetes.",
                "Machine learning engineer — training, experiments, PyTorch.",
            ],
        ),
        (
            "platform_sre",
            [
                "Platform/SRE engineer operating Kubernetes and improving reliability.",
                "Terraform, CI/CD, observability (Prometheus/Grafana), incident response/on-call.",
                "SLOs, alerts, runtime debugging, and infra automation.",
            ],
            [
                "Site reliability engineer — Kubernetes, Terraform, observability, on-call.",
                "Platform engineer — Kubernetes, CI/CD, monitoring.",
            ],
            [
                "DevOps engineer — CI/CD tooling only (limited k8s).",
                "Backend engineer — Python APIs + Docker (no on-call).",
                "Security engineer — appsec, threat modeling.",
            ],
        ),
        (
            "ml_embeddings",
            [
                "Software engineer working on embeddings, semantic search, and retrieval evaluation.",
                "Built batching/latency-aware pipelines and offline metrics; shipped to production.",
                "Strong Python; comfortable with APIs, Postgres, and pragmatic engineering.",
            ],
            [
                "Machine learning engineer — embeddings, retrieval, evaluation.",
                "Backend engineer — Python APIs + semantic search exposure.",
            ],
            [
                "Data scientist — notebooks, visualization, experiments (no production).",
                "Backend engineer — CRUD APIs only (no search/retrieval).",
                "Data engineer — Airflow/dbt pipelines.",
            ],
        ),
    ]

    generic_noise = [
        "We value clear communication and collaboration across teams.",
        "We ship incrementally and measure outcomes.",
        "We use code review and CI to maintain quality.",
        "We care about reliability, security, and privacy.",
        "We operate services in production and respond to incidents.",
    ]

    keyword_blends = [
        # intentionally cross-pollinate terms to create confusing near-misses
        ["Python", "Kubernetes", "React", "SQL"],
        ["FastAPI", "Airflow", "TypeScript", "Terraform"],
        ["Postgres", "Redis", "Spark", "Next.js"],
        ["Embeddings", "Retrieval", "Dashboards", "gRPC"],
    ]

    def resume_text(paras: list[str], blend: list[str]) -> str:
        extra = [
            "Projects:",
            "- Reduced latency via profiling and caching",
            "- Improved test reliability and deployment process",
            "- Wrote documentation and runbooks",
            "\nAmbiguous mentions (context varies): " + ", ".join(blend),
        ]
        return _long_text(paras + extra + rng.sample(generic_noise, k=3), repeat=3)

    def job_text(title: str, must: list[str], nice: list[str], blend: list[str]) -> str:
        parts = [
            f"Role: {title}",
            "About: Product team with mixed workloads and evolving requirements.",
            "Responsibilities: own features, write tests, collaborate, improve performance.",
            "Must-have:\n- " + "\n- ".join(must),
            "Nice-to-have:\n- " + "\n- ".join(nice),
            "Random stack mentions (not all are primary): " + ", ".join(blend),
        ]
        return _long_text(parts + rng.sample(generic_noise, k=2), repeat=2)

    cases = []
    for i in range(n_cases):
        name, r_paras, rel_titles, near_miss_titles = profiles[i % len(profiles)]
        blend = rng.choice(keyword_blends)
        resume = resume_text(r_paras, blend)

        cands = []

        # 2 relevant
        for j, title in enumerate(rel_titles[:2], start=1):
            must = ["Ownership", "Communication", "Shipping mindset"]
            nice = ["Documentation", "Mentoring"]
            # add a little domain match, but keep some ambiguity
            if name == "backend_python":
                must = ["Python", "APIs", "PostgreSQL"] + must
                nice = ["Redis", "Docker"] + nice
            elif name == "frontend_react":
                must = ["React", "TypeScript", "Accessibility"] + must
                nice = ["Next.js", "Testing"] + nice
            elif name == "data_engineer":
                must = ["SQL", "Python", "Pipelines"] + must
                nice = ["Airflow", "dbt", "Spark"] + nice
            elif name == "platform_sre":
                must = ["Kubernetes", "Terraform", "Observability"] + must
                nice = ["On-call", "SLOs"] + nice
            else:
                must = ["Embeddings", "Retrieval", "Evaluation", "Python"] + must
                nice = ["Batching", "Latency"] + nice

            text = job_text(title, must, nice, blend)
            cands.append({"id": f"{name}_rel_{i}_{j}", "text": text, "relevant": True})

        # many near-misses in same neighborhood
        while len(cands) < min(20, candidates_per_case - 5):
            title = rng.choice(near_miss_titles)
            must = ["Ownership", "Teamwork", "Quality"]
            nice = ["Some overlap with stack"]
            # intentionally include some matching keywords and some mismatching
            mixed = rng.sample(blend, k=min(2, len(blend))) + rng.sample(["Go", "Java", "Sales", "Figma", "Excel"], k=2)
            text = job_text(title, must + mixed, nice, mixed)
            cands.append({"id": f"{name}_nearmiss_{i}_{len(cands)}", "text": text, "relevant": False})

        # fill rest with broader negatives that still mention overlapping tech
        while len(cands) < candidates_per_case:
            other = rng.choice([p for p in profiles if p[0] != name])
            title = rng.choice(other[2] + other[3])
            mixed = rng.sample(blend, k=min(2, len(blend))) + rng.sample(["React", "Python", "SQL", "Kubernetes", "Airflow"], k=2)
            text = job_text(title, ["General engineering"] + mixed, ["Process"] + rng.sample(generic_noise, k=1), mixed)
            cands.append({"id": f"neg_{name}_{i}_{len(cands)}", "text": text, "relevant": False})

        rng.shuffle(cands)
        cases.append({"id": f"hard_{i:03d}_{name}", "resume_text": resume, "k": 5, "candidates": cands})

    return {"description": "Harder 120-case dataset with near-miss negatives and 50 candidates/case.", "cases": cases}


def eval_dataset(dataset: dict) -> dict:
    model = SentenceTransformer(MODEL_ID)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    per_case = []
    first_ranks = []

    for case in dataset["cases"]:
        resume = case["resume_text"].strip()
        candidates = case["candidates"]
        texts = [resume] + [c["text"].strip() for c in candidates]

        embs = model.encode(
            texts,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        r = embs[0:1]
        job = embs[1:]
        sims = (r @ job.T).squeeze(0)
        scores = [_sigmoid(float(sims[i].item())) for i in range(len(candidates))]

        order = sorted(range(len(candidates)), key=lambda i: scores[i], reverse=True)
        ranked_ids = [candidates[i]["id"] for i in order]
        pos = {c["id"] for c in candidates if c.get("relevant")}

        r1 = recall_at_k(ranked_ids, pos, 1)
        r3 = recall_at_k(ranked_ids, pos, 3)
        r5 = recall_at_k(ranked_ids, pos, 5)
        fr = first_relevant_rank(ranked_ids, pos)
        first_ranks.append(fr or 999999)

        per_case.append(
            {
                "case_id": case["id"],
                "recall@1": r1,
                "recall@3": r3,
                "recall@5": r5,
                "mrr": mrr(ranked_ids, pos),
                "first_relevant_rank": fr,
            }
        )

    mean_r1 = sum(x["recall@1"] for x in per_case) / len(per_case)
    mean_r3 = sum(x["recall@3"] for x in per_case) / len(per_case)
    mean_r5 = sum(x["recall@5"] for x in per_case) / len(per_case)
    mean_mrr = sum(x["mrr"] for x in per_case) / len(per_case)
    mean_first_rank = sum(x for x in first_ranks if x < 999999) / len([x for x in first_ranks if x < 999999])

    def bucket(rank: int | None) -> str:
        if rank is None:
            return "none"
        if rank == 1:
            return "1"
        if rank <= 3:
            return "2-3"
        if rank <= 5:
            return "4-5"
        if rank <= 10:
            return "6-10"
        if rank <= 20:
            return "11-20"
        return "21+"

    hist = Counter(bucket(x["first_relevant_rank"]) for x in per_case)

    return {
        "model": MODEL_ID,
        "device": device,
        "cases": len(per_case),
        "mean_recall@1": mean_r1,
        "mean_recall@3": mean_r3,
        "mean_recall@5": mean_r5,
        "mean_mrr": mean_mrr,
        "mean_first_relevant_rank": round(mean_first_rank, 3),
        "first_relevant_rank_hist": dict(hist),
        "worst10": sorted(per_case, key=lambda x: (x["mrr"], x["first_relevant_rank"] or 999999))[:10],
    }


hard = build_hard_dataset(n_cases=120, candidates_per_case=50, seed=2026)
OUT_PATH_HARD.write_text(json.dumps(hard, ensure_ascii=False, indent=2), encoding="utf-8")
print("wrote:", OUT_PATH_HARD)

summary = eval_dataset(hard)
print(json.dumps(summary, indent=2))


wrote: /home/uranium/personal/SeekersConnect/ai-tests/evals/dataset_hard_120.json


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 8225.38it/s]


{
  "model": "sentence-transformers/all-MiniLM-L6-v2",
  "device": "cpu",
  "cases": 120,
  "mean_recall@1": 0.4625,
  "mean_recall@3": 0.9208333333333333,
  "mean_recall@5": 0.9791666666666666,
  "mean_mrr": 0.9558333333333332,
  "mean_first_relevant_rank": 1.125,
  "first_relevant_rank_hist": {
    "2-3": 8,
    "1": 111,
    "4-5": 1
  },
  "worst10": [
    {
      "case_id": "hard_015_backend_python",
      "recall@1": 0.0,
      "recall@3": 0.0,
      "recall@5": 0.5,
      "mrr": 0.2,
      "first_relevant_rank": 5
    },
    {
      "case_id": "hard_000_backend_python",
      "recall@1": 0.0,
      "recall@3": 0.5,
      "recall@5": 0.5,
      "mrr": 0.3333333333333333,
      "first_relevant_rank": 3
    },
    {
      "case_id": "hard_090_backend_python",
      "recall@1": 0.0,
      "recall@3": 0.5,
      "recall@5": 1.0,
      "mrr": 0.3333333333333333,
      "first_relevant_rank": 3
    },
    {
      "case_id": "hard_115_backend_python",
      "recall@1": 0.0,
      "recall

In [7]:
# Quick resume/job similarity sanity check
# using sentence-transformers/all-MiniLM-L6-v2.

import json
import math

import torch
from sentence_transformers import SentenceTransformer


def _sigmoid(x: float) -> float:
    if x >= 0:
        z = math.exp(-x)
        return 1.0 / (1.0 + z)
    z = math.exp(x)
    return z / (1.0 + z)


MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_ID)

resume = "Senior software engineer. Strong Python, FastAPI, PostgreSQL, Docker."
job_good = "Backend engineer — Python, APIs, Postgres, AWS."
job_bad = "Line cook for busy downtown restaurant. Nights and weekends."

embs = model.encode(
    [resume, job_good, job_bad],
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

sim_good = float((embs[0:1] @ embs[1:2].T).item())
sim_bad = float((embs[0:1] @ embs[2:3].T).item())

print(
    json.dumps(
        {
            "model": MODEL_ID,
            "device": "cuda" if torch.cuda.is_available() else "cpu",
            "cosine_similarity": {"good": sim_good, "bad": sim_bad},
            "sigmoid_score": {"good": _sigmoid(sim_good), "bad": _sigmoid(sim_bad)},
        },
        indent=2,
    )
)


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 7731.37it/s]


{
  "model": "sentence-transformers/all-MiniLM-L6-v2",
  "device": "cpu",
  "cosine_similarity": {
    "good": 0.7301715612411499,
    "bad": 0.1919725239276886
  },
  "sigmoid_score": {
    "good": 0.6748429193866597,
    "bad": 0.5478462794525483
  }
}


In [ ]:
# Optional: run the matcher HTTP service (FastAPI)
#
# Note:
# - The current `ai-tests/matcher_service.py` is still wired to the older
#   BGE+LoRA embedding implementation.
# - This notebook is being updated first to validate MiniLM.
# - After the notebook is confirmed working, we can port the same change into
#   `ai-tests/` and then re-enable this cell.

print("Skipping service start for now (ai-tests/matcher_service.py not yet ported to MiniLM).")
